In [ ]:
import numpy as np
import pandas as pd

# Display settings for clean terminal output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("=" * 80)
print("  🎮 ESPORTS MATCHMAKING & ANTI-CHEAT SYSTEM (NumPy + Pandas Pipeline)")
print("=" * 80)

# =============================================================================
# STEP 1: Create the Raw Match Dataset
# Scenario: Post-game telemetry data with missing fields and suspicious numbers
# =============================================================================
raw_data = {
    'player_id': ['Ninja99', 'GhostRider', 'NoobMaster', 'ViperX', 'AimbotGod', 
                  'ShadowFox', 'CyberSamurai', 'PixelQueen', 'RageQuitter', 'LaggyBoy'],
    'kills': [18, 5, 2, 22, 95, 12, np.nan, 8, 1, 14],             # 95 is impossible (Potential Cheater)
    'deaths': [3, 8, 12, 4, 1, 5, 6, 9, 15, np.nan],               # Missing death stat
    'headshot_rate': [0.42, 0.21, 0.08, 0.48, 0.99, 0.35, 0.28, np.nan, 0.05, 0.30], # 0.99 is suspicious
    'ping_ms': [25, 45, 60, 30, 20, 38, 55, 42, 180, 220]          # Network latency in milliseconds
}

df = pd.DataFrame(raw_data)
print("\n[Raw In-Game Telemetry Data]")
print(df)

# =============================================================================
# STEP 2: Intelligent Missing Data Imputation
# Concept: Fill missing numbers using statistical aggregations (Median & Mean)
# =============================================================================

# Calculate robust median for kills (resilient to cheater outliers)
clean_kills_median = np.nanmedian(df['kills'])
# Calculate standard average for headshot accuracy
clean_hs_mean = np.nanmean(df['headshot_rate'])

# Impute values directly using Pandas + NumPy
df['kills'] = df['kills'].fillna(clean_kills_median)
df['headshot_rate'] = np.round(df['headshot_rate'].fillna(clean_hs_mean), 2)
# If deaths are missing, assume at least 1 to avoid division by zero later
df['deaths'] = df['deaths'].fillna(1)

# =============================================================================
# STEP 3: Vectorized Feature Engineering
# Concept: Element-wise arithmetic without any slow 'for' loops
# =============================================================================

# 1. Kill-to-Death Ratio (KD): np.maximum ensures denominator is never zero
df['kd_ratio'] = np.round(df['kills'] / np.maximum(df['deaths'], 1), 2)

# 2. Combat Effectiveness Score: Combining Kills and Headshot Accuracy
df['combat_power'] = np.round((df['kills'] * 1.5) + (df['headshot_rate'] * 100), 2)

# =============================================================================
# STEP 4: Anti-Cheat & Anomaly Detection (IQR Rule)
# Concept: Flag players with inhuman combat stats using Interquartile Range
# =============================================================================

q25 = np.percentile(df['combat_power'], 25)
q75 = np.percentile(df['combat_power'], 75)
iqr = q75 - q25
suspicious_threshold = q75 + (1.5 * iqr)

# np.where: If combat_power exceeds threshold, tag as Potential Cheater
df['anti_cheat_flag'] = np.where(df['combat_power'] > suspicious_threshold, True, False)

# Winsorization: Cap extreme performance to keep balanced metrics
df['combat_power_capped'] = np.clip(df['combat_power'], a_min=None, a_max=suspicious_threshold)

# =============================================================================
# STEP 5: Min-Max Feature Normalization (0.0 to 1.0)
# Concept: Prepare raw power into a normalized index for the ranking algorithm
# =============================================================================

min_cp = np.min(df['combat_power_capped'])
max_cp = np.max(df['combat_power_capped'])

# Normalization Formula: (X - min) / (max - min)
df['normalized_skill'] = np.round((df['combat_power_capped'] - min_cp) / (max_cp - min_cp), 3)

# =============================================================================
# STEP 6: Multi-Condition Logic for Match Tiering
# Concept: Fast categorization using np.select
# =============================================================================

# Define operational rules based on skill and network latency (Ping)
conditions = [
    (df['anti_cheat_flag'] == True),                                        # Flagged for hacking
    (df['ping_ms'] > 150),                                                  # High packet delay / lag
    (df['normalized_skill'] >= 0.75) & (df['ping_ms'] <= 50),               # Elite competitive lobby
    (df['normalized_skill'] >= 0.35) & (df['normalized_skill'] < 0.75),     # Mainstream competitive
]

tier_assignments = [
    'BANNED_PENDING_REVIEW',
    'HIGH_PING_CONTAINMENT',
    'TIER_1_PRO_LOBBY',
    'TIER_2_GOLD_LOBBY'
]

# Default bucket for remaining entry-level or casual players
df['assigned_lobby'] = np.select(conditions, tier_assignments, default='TIER_3_CASUAL_LOBBY')

# =============================================================================
# STEP 7: Final Analytics Dashboard Output
# =============================================================================

display_columns = [
    'player_id', 'kd_ratio', 'combat_power', 
    'normalized_skill', 'anti_cheat_flag', 'assigned_lobby'
]

print("\n[Processed Matchmaking & Security Dashboard]")
print(df[display_columns])

# Summary metrics using pure NumPy
print("\n" + "-" * 80)
print("📊 SERVER REAL-TIME TELEMETRY:")
print(f"• Average Lobby Skill Index : {np.mean(df['normalized_skill']):.3f}")
print(f"• Security Incident Count   : {np.sum(df['anti_cheat_flag'])} detected account(s)")
print(f"• Server Health Status      : {np.where(np.mean(df['ping_ms']) < 80, 'OPTIMAL', 'DEGRADED')}")
print("-" * 80)


  🎮 ESPORTS MATCHMAKING & ANTI-CHEAT SYSTEM (NumPy + Pandas Pipeline)

[Raw In-Game Telemetry Data]
      player_id  kills  deaths  headshot_rate  ping_ms
0       Ninja99   18.0     3.0           0.42       25
1    GhostRider    5.0     8.0           0.21       45
2    NoobMaster    2.0    12.0           0.08       60
3        ViperX   22.0     4.0           0.48       30
4     AimbotGod   95.0     1.0           0.99       20
5     ShadowFox   12.0     5.0           0.35       38
6  CyberSamurai    NaN     6.0           0.28       55
7    PixelQueen    8.0     9.0            NaN       42
8   RageQuitter    1.0    15.0           0.05      180
9      LaggyBoy   14.0     NaN           0.30      220

[Processed Matchmaking & Security Dashboard]
      player_id  kd_ratio  combat_power  normalized_skill  anti_cheat_flag         assigned_lobby
0       Ninja99      6.00          69.0             0.586            False      TIER_2_GOLD_LOBBY
1    GhostRider      0.62          28.5             0